# Rýchla práca so všetkými premennými v Parquete

Plný súbor `student_full.parquet` obsahuje všetkých žiakov a všetky premenné
z pôvodného SAV. Pre novú tému ho nemusíme upravovať ani vytvárať znova.
V každej analýze si vyberieme len stĺpce a riadky, ktoré chceme načítať.

Pred prvým spustením priprav dáta podľa README. K Parquetu patrí aj
`student_full_metadata.json` s popismi premenných a významom kódov zo SAV.
Tento notebook už pôvodný SAV nečíta a žiadne dáta nemení.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

DATA = PROJECT / "data/processed/student_full.parquet"
METADATA = PROJECT / "data/processed/student_full_metadata.json"
if not DATA.is_file() or not METADATA.is_file():
    raise FileNotFoundError("Najprv priprav plný Parquet podľa kroku 5 v README.")

metadata = json.loads(METADATA.read_text(encoding="utf-8"))
parquet = pq.ParquetFile(DATA)
if json.loads(parquet.schema_arrow.metadata[b"pisa_metadata"]) != metadata:
    raise ValueError("Parquet a metadata JSON nepatria k sebe. Použi oba súbory z rovnakého prevodu.")
print(f"Záznamy: {parquet.metadata.num_rows:,}")
print(f"Premenné: {parquet.metadata.num_columns}")

## Vyhľadávanie premenných

Na prehľad názvov a významov stačia metadáta; žiacke záznamy zatiaľ nečítame.
Zmeň `SEARCH` podľa toho, čo chceš skúmať.


In [ ]:
variables = pd.DataFrame(metadata["columns"])[["name", "label", "readstat_type"]]
SEARCH = "artificial intelligence"
mask = variables["name"].str.contains(SEARCH, case=False, regex=False)
mask |= variables["label"].str.contains(SEARCH, case=False, regex=False, na=False)
variables.loc[mask]

## Načítanie vybraných stĺpcov a krajín

Toto je ukážka výberu premenných k téme AI. Uprav zoznam `COLUMNS` a krajiny
v `COUNTRIES`; spoločný Parquet sa tým nemení. Filter krajiny obmedzí načítané
riadky a `columns` obmedzí načítané stĺpce. Načítanie všetkých 885 premenných
naraz by zabralo podstatne viac pamäte než veľkosť komprimovaného súboru.


In [ ]:
COLUMNS = ["CNT", "CNTSTUID", "ESCS", "ST438Q04DA"]
COUNTRIES = ["CZE", "SVK"]

df = pd.read_parquet(
    DATA,
    columns=COLUMNS,
    filters=[("CNT", "in", COUNTRIES)],
)
print(f"Načítané: {len(df):,} riadkov × {len(df.columns)} stĺpcov")
df.head(20)

## Význam kódov a chýbajúce odpovede

Parquet zachováva pôvodné číselné kódy vrátane špeciálnych kódov pre
chýbajúce odpovede. Nič sa automaticky nečistí ani nedopĺňa. Napríklad
v otázkach o AI je `5` platná odpoveď o dennom používaní, zatiaľ čo `95–99`
majú osobitné významy. Spôsob čistenia určíme pre konkrétne premenné.

Popisy nižšie sú prevzaté zo SAV. Pri interpretácii používaj aj dodaný
OECD codebook v `data/reference/`.


In [ ]:
VARIABLE = "ST438Q04DA"
by_name = {column["name"]: column for column in metadata["columns"]}
print(by_name[VARIABLE]["label"])
pd.DataFrame(by_name[VARIABLE]["value_labels"])

## Ďalšia analýza

Pre inú tému alebo ďalšiu krajinu meníme výber pri načítaní. Prevod SAV
opakujeme až pri zmene pôvodných dát. Výberové váhy, replikačné váhy aj všetky
plausible values sú v plnom Parquete dostupné; pri analýze ich treba
cielene vybrať a správne použiť.
